In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BusDisruptionProject")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("PySpark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/16 10:56:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark version: 3.5.1


## libraries

In [6]:
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
import os
import requests
import xml.etree.ElementTree as ET

print("Libraries imported successfully.")

Libraries imported successfully.


## project folder

In [7]:
current_folder = Path.cwd()
if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder
print("Current folder:", current_folder)
print("Project root:", project_root)

Current folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/notebooks
Project root: /Users/babitaadhikari/Desktop/bus-disruption-platform


## vehicle-location folder

In [8]:
location_folder = (
    project_root
    / "data"
    / "raw"
    / "vehicle_location"
)
location_folder.mkdir(
    parents=True,
    exist_ok=True
)
print("Vehicle-location folder:")
print(location_folder)

Vehicle-location folder:
/Users/babitaadhikari/Desktop/bus-disruption-platform/data/raw/vehicle_location


In [9]:
env_path = project_root / ".env"

loaded = load_dotenv(env_path, override=True)

if not loaded:
    raise FileNotFoundError(
        f".env file was not found at: {env_path}"
    )

api_key = os.getenv("BODS_API_KEY")
feed_id = os.getenv("BODS_FEED_ID", "10609")

if not api_key:
    raise ValueError(
        "BODS_API_KEY is missing from the .env file."
    )

feed_url = (
    f"https://data.bus-data.dft.gov.uk/"
    f"api/v1/datafeed/{feed_id}/"
)

print("API configuration loaded securely.")
print("Feed ID:", feed_id)

API configuration loaded securely.
Feed ID: 10609


## the feed URL

In [10]:
import requests

try:
    response = requests.get(
        feed_url,
        params={"api_key": api_key},
        timeout=120
    )

    print("Status code:", response.status_code)
    print(
        "Content type:",
        response.headers.get("Content-Type", "Unknown")
    )
    print("Downloaded bytes:", len(response.content))

    response.raise_for_status()

except requests.RequestException as error:
    raise RuntimeError(
        f"Download failed: {error}"
    )

Status code: 200
Content type: text/xml
Downloaded bytes: 1381970


In [11]:
content_start = response.content.lstrip()[:100]

if not content_start.startswith(b"<"):
    print(response.text[:500])
    raise ValueError(
        "The downloaded response is not XML."
    )

print("The downloaded response appears to be XML.")

The downloaded response appears to be XML.


## save the XML file

In [12]:
timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

output_file = location_folder / (
    f"nx_west_midlands_{timestamp}.xml"
)

output_file.write_bytes(
    response.content
)

print("Vehicle-location data downloaded successfully.")
print("Saved file:", output_file)
print(
    "File size:",
    round(
        output_file.stat().st_size / 1_000_000,
        2
    ),
    "MB"
)

Vehicle-location data downloaded successfully.
Saved file: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/raw/vehicle_location/nx_west_midlands_20260716T051150Z.xml
File size: 1.38 MB


In [13]:
try:
    tree = ET.parse(output_file)
    root = tree.getroot()

    print("XML parsed successfully.")
    print("Root element:", root.tag)

except ET.ParseError as error:
    raise ValueError(
        f"Invalid XML file: {error}"
    )

XML parsed successfully.
Root element: {http://www.siri.org.uk/siri}Siri


In [14]:
def remove_namespace(tag):
    return tag.split("}")[-1]


vehicle_records = [
    element
    for element in root.iter()
    if remove_namespace(element.tag)
    == "VehicleActivity"
]

print(
    "VehicleActivity records:",
    len(vehicle_records)
)

VehicleActivity records: 1236


In [15]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


## automatic snapshot collection

this section collects vehicle location snapshots every ten minutes

In [16]:
# import time for the waiting interval

import time

# collect one snapshot every ten minutes

snapshot_interval_seconds = 600

# collect twenty four snapshots

number_of_snapshots = 24

print(
    "snapshot interval:",
    snapshot_interval_seconds / 60,
    "minutes"
)

print(
    "number of snapshots:",
    number_of_snapshots
)

print(
    "estimated collection time:",
    number_of_snapshots * 10,
    "minutes"
)

snapshot interval: 10.0 minutes
number of snapshots: 24
estimated collection time: 240 minutes


In [17]:
# import required libraries

import time
import requests

from datetime import datetime, timezone

In [ ]:
# collect vehicle location snapshots automatically

successful_snapshots = 0
failed_snapshots = 0

for snapshot_number in range(
    1,
    number_of_snapshots + 1
):
    print(
        "collecting snapshot",
        snapshot_number,
        "of",
        number_of_snapshots
    )

    try:
        response = requests.get(
            feed_url,
            params={
                "api_key": api_key
            },
            timeout=120
        )

        response.raise_for_status()

        content_start = (
            response.content
            .lstrip()[:100]
        )

        if not content_start.startswith(b"<"):
            raise ValueError(
                "the downloaded response is not xml"
            )

        timestamp = datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )

        output_file = (
            location_folder
            /
            f"nx_west_midlands_{timestamp}.xml"
        )

        output_file.write_bytes(
            response.content
        )

        successful_snapshots += 1

        print(
            "saved:",
            output_file.name
        )

        print(
            "file size:",
            round(
                output_file.stat().st_size
                /
                1_000_000,
                2
            ),
            "MB"
        )

    except Exception as error:
        failed_snapshots += 1

        print(
            "snapshot failed:",
            error
        )

    if snapshot_number < number_of_snapshots:
        print(
            "waiting ten minutes"
        )

        time.sleep(
            snapshot_interval_seconds
        )

print(
    "collection completed"
)

print(
    "successful snapshots:",
    successful_snapshots
)

print(
    "failed snapshots:",
    failed_snapshots
)

collecting snapshot 1 of 24
saved: nx_west_midlands_20260716T051152Z.xml
file size: 1.38 MB
waiting ten minutes
collecting snapshot 2 of 24
saved: nx_west_midlands_20260716T062218Z.xml
file size: 1.39 MB
waiting ten minutes


In [20]:
# count all saved xml snapshots

xml_files = sorted(
    location_folder.glob(
        "*.xml"
    )
)

print(
    "total xml snapshots:",
    len(xml_files)
)

print(
    "latest snapshots:"
)

for xml_file in xml_files[-10:]:
    print(
        xml_file.name
    )

total xml snapshots: 34
latest snapshots:
nx_west_midlands_20260716T083239Z.xml
nx_west_midlands_20260716T084241Z.xml
nx_west_midlands_20260716T085243Z.xml
nx_west_midlands_20260716T090244Z.xml
nx_west_midlands_20260716T091246Z.xml
nx_west_midlands_20260716T092247Z.xml
nx_west_midlands_20260716T093249Z.xml
nx_west_midlands_20260716T094250Z.xml
nx_west_midlands_20260716T095252Z.xml
nx_west_midlands_20260716T100254Z.xml
